In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_30_try_0.pickle

trying: ['title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  21
me:  6
me:  17
me:  8
me:  10
trying: ['title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  21
me:  6
me:  17
me:  8
me:  10
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  16
me:  16
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  16
me:  16
trying: ['question_new', 'question_of_interest_cell40']
me:  28
me:  28
trying: ['question_new', 'question_of_interest_cell40']
me:  28
me:  28
trying: ['question_orig_2019', 'question_orig_2021']
me:  28
me:  28
trying: ['question_orig_2019', 'question_orig_2021']
me:  28
me:  28
trying: ['title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  21
me:  6
me:  17
me:  8
me:  10
trying: ['title_for_x_axis_cell33', 'title_fo

me:  0
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35']
me:  23
trying: ['sns']
me:  0
trying: ['responses_df_2017']
me:  17
trying: ['language_df_combined_counts']
me:  23
trying: ['warnings']
me:  0
trying: ['grab_subset_of_data_35']
me:  23
trying: ['go']
me:  0
trying: ['responses_df_2019_cell10']
me:  21
trying: ['language_df_combined']
me:  23
trying: ['language_df_combined_percentages_cell35']
me:  23
trying: ['df']
me:  14
trying: ['language_df_combined_percentages']
me:  23
trying: ['convert_df_of_counts_to_percentages_35']
me:  23
trying: ['grab_subset_of_data_39']
me:  27
trying: ['add_year_column_to_dataframes_35']
me:  23
trying: ['definitions']
me:  27
trying: ['df_cell35']
me:  23
trying: ['notebooks_df_2022']
me:  27
trying: ['question_of_interest_cell35']
me:  23
trying: ['responses_df_2020']
me:  21
trying: ['responses_df_2018_cell10']
me:  26
trying: ['responses_df_2021']


me:  17
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['np']
me:  0
trying: ['learning_platform_df_combined_counts']
me:  14
trying: ['learning_platform_df_combined_percentages_cell26']
me:  14
trying: ['question_of_interest_alternate']
me:  14
trying: ['question_of_interest_cell26']
me:  14
trying: ['grab_subset_of_data_26']
me:  14
trying: ['add_year_column_to_dataframes_26']
me:  14
trying: ['convert_df_of_counts_to_percentages_26']
me:  14
trying: ['learning_platform_df_combined']
me:  14
trying: ['df_cell26']
me:  14
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26']
me:  14
trying: ['learning_platform_df_combined_percentages']
me:  14
trying: ['question_of_interest_cell33']
me:  21
trying: ['combine_subset_of_data_from_multiple_years_33']
me:  21
trying: ['responses_df_2018']
me:  1
trying: ['add_year_column_to_dataframes_33']
me:  21
trying: ['count_then_return_percent_33']
me:  21
trying: ['subset_of_countries']
me

In [3]:
%%RecordEvent
%%time
### cell 31 ###

# Rename and unify response categories for 2018 and 2019
question_of_interest_cell43 = 'For how many years have you used machine learning methods?'
old_col_2018 = 'For how many years have you used machine learning methods (at work or in school)?'
responses_df_2018_cell10.rename(columns={old_col_2018: question_of_interest_cell43}, inplace=True)

responses_df_2018_cell10.replace({question_of_interest_cell43: {
    '< 1 year': 'Under 1 year',
    '10-15 years': '10-20 years',
    '20+ years': '10-20 years',
    'I have never studied machine learning but plan to learn in the future': 'I do not use machine learning methods',
    'I have never studied machine learning and I do not plan to': 'I do not use machine learning methods'
}}, inplace=True)

responses_df_2019_cell10.replace({question_of_interest_cell43: {
    '< 1 years': 'Under 1 year',
    '10-15 years': '10-20 years',
    '20+ years': '20 or more years'
}}, inplace=True)

# Prepare x-axis title (empty here)
title_for_x_axis_cell43 = ''

# Optimized percentage counter
def count_then_return_percent_43(df, col):
    counts = df[col].value_counts(dropna=False)
    total = df[col].count()
    return (counts * 100 / total).round(1)

# Optimized combiner
def combine_subset_of_data_from_multiple_years_43(question, x_axis_title, include_2017=None):
    sources = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10)
    ]
    if include_2017 is not None:
        sources.append(('2017', responses_df_2017))
    df_list = []
    for year, src in sources:
        ser = count_then_return_percent_43(src, question).sort_index()
        tmp = ser.rename_axis(x_axis_title).reset_index(name='percentage')
        tmp['year'] = year
        df_list.append(tmp)
    combined = pd.concat(df_list, ignore_index=True, sort=False)
    # ensure column order is ['percentage','year', x_axis_title]
    return combined[['percentage', 'year', x_axis_title]]

# Build and inspect the combined DataFrame
ml_exp_df_combined = (
    combine_subset_of_data_from_multiple_years_43(
        question_of_interest_cell43,
        title_for_x_axis_cell43
    )
    .sort_values(['year', 'percentage'], ascending=True)
)
ml_exp_df_combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48 entries, 40 to 8
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  48 non-null     float64
 1   year        48 non-null     object 
 2               43 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.5+ KB
CPU times: user 16.8 ms, sys: 106 µs, total: 16.9 ms
Wall time: 16.9 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_31_try_1.pickle

migration speed (bps): 735361870.9635847


---------------------------
variables to migrate:
title_for_y_axis_cell24 65
np 72
question_of_interest_cell24 116
India_responses_df_2022 9195445
grab_subset_of_data_37 144
bar_chart_multiple_choice_24 144
question_of_interest_cell37 146
USA_responses_df_2022 3060721
columns_to_combine 120
count_then_return_percent_24 144
ide_df_2022_cell37 978389
percentages_cell24 151
columns_to_combine_cell37 120
percentages_cell42 642
responses_in_order_cell42 120
question_of_interest_cell42 107
count_then_return_percent_42 144
age_df_combined 10611
incorrect_phrasing 64
ide_df_combined 10725491
ide_df_combined_percentages 1209
alternate_phrasing 168
responses_in_order_cell28 152
ide_df_combined_percentages_cell38 649
count_then_return_percent_28 144
correct_phrasing 78
percentages_cell28 561
x_axis_title 74
responses_df_2022 25378603
responses_df_2022_cell10 25378603
ide_df_combined_counts_2 1209
percentages_cell17 958
question_of_interest_cell28 160
ide_df_combined_2_cell38 7399408
responses_in_

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2018', 'responses_df_2019', 'responses_df_2022', 'responses_df_2020', 'responses_df_2021']
Intermediate variables ['file_name_2021', 'base_dir_2017', 'factor', 'responses_df_2017', 'file_name_2022', 'file_name_2017', 'file_name_2018', 'file_name_2019', 'base_dir_2018', 'directory_cell8', 'base_dir_2022', 'base_dir_2019', 'base_dir_2021', 'file_name_2020', 'directory', 'base_dir_2020']
Future variables ['responses_df_2018_cell10', 'responses_df_2017', 'percentages', 'responses_df_2022_cell10']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2022', 'responses_df_2018', 'responses_df_2019']
Active variables ['responses_df_2022', 'responses_df_2019_cell10', 'responses_df_2022_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_d

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_31_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[31], f)


In [7]:
opt_output = Out.get(4)

In [ ]:
assert False, 'title_for_x_axis_cell43 is incorrectly modified in the optimized code.'